# Using Paicos actions for movie generation

This is a brief example of using the new Paicos Actions class to generate movies.
I'd really appreciate any feedback on this!

This notebooks currently (2/5/2024) requires that you check out the `actions`
branch of Paicos. That is, you need to do:

```
git checkout actions
```
before following the usual instructions for a developer installation (https://paicos.readthedocs.io/en/latest/installation.html#option-2-developer-installation-compile-the-code-and-add-its-path-to-your-pythonpath).

Alternatively, you can install directly from the `actions` branch:
```
pip install git+https://github.com/tberlok/paicos.git@actions
```

In [1]:
import numpy as np
import paicos as pa
pa.settings.strict_units = False
import os
import glob
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import turbocluster as tc



The default number of OpenMP threads, 8, exceeds the 1 available on your system. Setting numthreads=1. You can set numthreads with e.g. the command
 paicos.set_numthreads(16)




In [2]:
data_dir = '/llust21/cosmo-plasm/zoom-simulations-arepo2/halo_0003/tng/zoom12/output/'
snapnums = [305]
# snap = pa.Snapshot('/llust21/cosmo-plasm/zoom-simulations-arepo2/halo_0003/tng/zoom12/output', 305, 
# basename='snapshot')
# center = snap.Cat.Group['GroupPos'][0]
# widths = np.array([1e3, 1e3, 1e3], dtype=float)

# m_filter = 1000*snap.mass
# filter_length = (np.cbrt(3*m_filter/(4*np.pi*snap['0_Density']))).arepo
# filter_length = 2.0*snap['0_Diameters']
weight='0_Volume'
filter_type = 'gaussian'

In [3]:
# data_dir = '/llust21/joseph/181022_lmp/output/'
# files = sorted(glob.glob(data_dir + 'snapshot_*.hdf5'))
# files = files[-10:]
# snapnums = [int(file[-8:-5]) for file in files]

## Write your own subclass of the Paicos Actions class

Your subclass should set the 'image_creator' for a given input snapnum.

There are two if-clauses:

* When `first_call = True` is the very first view that we manually specify,
subsequent views will here inherit the orientation, widths and pixel dimensions (you could modify
to also inherit the center, for this particular simulation the center of the cluster is moving
through the box and so I load the center for each newly loaded snapshot).

* with `dry_run=True`, we are initializing the Base class ImageCreator, which is computationally
  cheaper than initializing the full Projector class.

In [4]:
class Actions(pa.Actions):
    def make_image_creator(self, snapnum):

        snap = pa.Snapshot(data_dir, snapnum, basename='snapshot', load_catalog=False)
        if snapnum >= 100:
            cat = pa.Catalog(data_dir, snapnum)
            center = cat.Group['GroupPos'][0]
        else:
            cat = pa.Catalog(data_dir, 100)
            center = cat.Group['GroupPos'][0].value * snap.length

        if self.first_call:
            widths = [2000., 2000., 1000.]

            # Set the widths of the image so that we can make movies with 1920x1080 resolution
            widths[0] = 1920 / 1080 * widths[1]
            
            # Pixel dimensions of image
            npix = 1920
            orientation = pa.Orientation(normal_vector=[0, 0, 1], perp_vector1=[1, 0, 0])

        else:
            orientation = self.image_creator.orientation.copy
            npix = self.image_creator.npix

            widths = self.image_creator.widths.value * snap.length
            del self.image_creator

        if self.dry_run:
            self.image_creator = pa.ImageCreator(snap, center, widths, orientation, npix=npix)
        else:
            self.image_creator = pa.Projector(snap, center, widths, orientation, npix=npix, make_snap_with_selection=False)

## Set up a couple of functions for creating hdf5s, pngs and interactive figures

Unlike the `make_image_creator` method above, Paicos does not require that you write the functions below. So this part is just for inspiration and to have a complete example.

In [11]:
def make_hdf5(actions, verbose=True):
    projector = actions.image_creator
    image_file = pa.ImageWriter(projector, basedir=actions.outfolder + 'hdf5s',
                               basename=f'{actions.basename}_frame_{actions.frame_num}')

    masses = projector.project_variable('0_Masses')
    volumes = projector.project_variable('0_Volume')
    rho = masses / volumes

    image_file.save_image('0_Density', rho)

    # Move from temporary filename to final filename
    image_file.finalize()
    print(f'frame {actions.frame_num} is done\n')

def make_plot(frame, interactive=False):

    if not interactive:
        # if not os.path.exists(files[frame]):
        #     # print('file still missing')
        #     return

        if not os.path.exists('./../../test-actions/pngs/'):
            os.makedirs('./../../test-actions/pngs/')
        
        
    # if os.path.exists(f'./images_1920_{frame}.png'):
    #     # print('skipping\n')
    #     return
    
    my_dpi = 192
    fig = plt.figure(figsize=(1920 / my_dpi, 1080 / my_dpi), dpi=my_dpi)
    # fig.set_size_inches(w,h)
    ax = plt.Axes(fig, [0., 0., 1., 1.])
    ax.set_axis_off()
    fig.add_axes(ax)

    if not interactive:
        im = pa.ImageReader(files[frame])
        # im = pa.ImageReader(basedir=actions.outfolder + 'hdf5s', snapnum=frame, basename='snapshot')
        # im = pa.ImageReader(basedir=actions.outfolder + 'hdf5s', 
        #                basename=f'{actions.basename}_frame_{actions.frame_num}_{frame}')


        extent = im.centered_extent.to_physical.to('kpc')
        rho = im['0_Density'].to_physical.cgs
    else:
        projector = im = actions.image_creator
        im.age = projector.snap.age
        extent = projector.centered_extent.to_physical.to('kpc')

        masses = projector.project_variable('0_Masses')
        volumes = projector.project_variable('0_Volume')
        rho = (masses / volumes).to_physical.cgs

    image = plt.imshow(rho.value, origin='lower', norm=LogNorm(), extent=extent.value, cmap='viridis')

    
    sl = im.width.to_physical.to('kpc')

    candidates = [0.01, 0.1, 1, 10, 20, 50, 100, 200, 500, 1000]

    index = np.argmin(np.abs(0.05*sl.value - candidates))
    marker_length = candidates[index]
    extent = extent.value

    anno_opts = dict(xy=(0.08, 0.8), xycoords='axes fraction',
                     va='center', ha='left', fontsize=20)
    # ax.annotate(f'{im.age.value} {im.age.label()}', **anno_opts)
    ax.annotate(f'{im.age:1.2f}', **anno_opts)
    
    anno_opts = dict(xy=(0.08, 0.1), xycoords='axes fraction',
                     va='center', ha='left', fontsize=20)
    ax.annotate(f'{marker_length} kpc', **anno_opts)

    ax.plot([extent[0]+sl.value/10, extent[0] + sl.value/10 + marker_length],
                    [extent[2] + sl.value/10, extent[2] + sl.value/10], 'k-', lw=3)

    cbaxes = fig.add_axes([0.67, 0.075, 0.3, 0.05])
    cbar = plt.colorbar(cax=cbaxes, orientation='horizontal', ticklocation='bottom')
    cbar.ax.tick_params(labelsize=15)

    anno_opts = dict(xy=(0.67 + 0.3/2, 0.2), xycoords='axes fraction',
                     va='center', ha='center', fontsize=20)
    ax.annotate(rho.label(r'\rho'), **anno_opts)
    # cbar.set_label(label=rho.label(r'\rho'), size=15)

    if not interactive:
        fig.savefig(f'./../../test-actions/pngs/images_1920_{frame}.png', dpi=my_dpi)
        print(f'image {frame} done')
        plt.close(fig)
    else:
        plt.show()


# Make a crude plan for movie

We are now ready to make a crude plan for the movie. Below we first loop through all the
available snapshots. Then we rotate 180 degrees around the horizontal axis, followed by a rotation by 180 degrees around the vertical axis of the image. We then zoom in by a factor of 10, and do the rotations in the opposite order and direction.

In [6]:
# Make some initial plan for the movie
actions = Actions(snapnum=305)
actions.create_log(outfolder='./../../test-actions/', basename='image')
# for snapnum in snapnums:
#     actions.change_snapnum(snapnum)

# actions.rotate_around_perp_vector1(180)
# actions.rotate_around_perp_vector2(180)

actions.zoom(10.)

actions.rotate_around_perp_vector2(180)
# actions.rotate_around_perp_vector1(180)

# Expanding the logfile

The movie above would look very jumpy. Below we use the `expand_log` method to insert incremental steps, so that we at most zoom in by a factor 1.02 and at most rotate by an angle 1 degree. This creates a new logfile `image_expanded.log`

In [7]:
actions = Actions()
actions.read_log(outfolder='./../../test-actions/', basename='image')
actions.expand_log(zoom_max=2, max_angle=10, verbose=False)

self.image_creator not set, you'll nedd to do this manually or read a log file using Actions.read_log


In [ ]:
# This is how to resume from frame number 14
# actions.resume_from_waypoint(14)

# Make a movie using Actions and a log-file

We are now ready to use `image_expanded.log` to generate a all the hdf5s that we will then turn into a movie.
We therefore set `dry_run=False` below.

In [8]:
actions = Actions(dry_run=False)
# actions.read_log(outfolder='test-actions', basename='image')
actions.read_log(outfolder='./../../test-actions', basename='image_expanded')

self.image_creator not set, you'll nedd to do this manually or read a log file using Actions.read_log
Attempting to get derived variable: 0_Volume...	[DONE]



## Illustration of interactively changing the view

Before we start the movie-making, we take a quick look at the galaxy. You can play around as much as you want, as the final command resets the view.

In [ ]:
# actions.zoom(0.3)
# # actions.rotate_around_normal_vector(100)
# actions.change_snapnum(60)
# actions.move_center(np.array([-1000, 0, 0])*actions.image_creator.snap.length)
# actions.rotate_around_normal_vector(-30)
# make_plot(None, interactive=True)

# # Change back to original view at frame_num 0
# actions.resume_from_waypoint(0)

## Loop over all steps in the logfile

Let's start the computation! I included a bash cell for installing `tqdm` in case you don't already have it installed.

In [ ]:
# %%bash
# pip install tqdm

In [ ]:
from tqdm.notebook import tqdm
make_hdf5(actions)
for _ in tqdm(range(actions.num_steps)):
    print(actions.image_creator.snap.snapnum, actions.image_creator.orientation, actions.image_creator.widths)
    actions.step(verbose=True)
    make_hdf5(actions, verbose=True)

## Make pngs from hdf5s
First we find the hdfs files, and the corresponding snapnums and framenums.

In [9]:
# Find the hdf5 filenames sorted by framenumber
# files = glob.glob('./test-actions/hdf5s/image_frame_*_*.hdf5')
files = glob.glob('./../../test-actions/hdf5s/image_expanded_frame_*_*.hdf5')
framenums = [int(file.split('_')[-2]) for file in files]
snapnums = [int(file[-8:-5]) for file in files]
index = np.argsort(framenums)
files = [files[ii] for ii in index]
snapnums = np.array(snapnums)[index]
framenums = np.array(framenums)[index]

In [10]:
files

['./test-actions/hdf5s/image_expanded_frame_0_305.hdf5',
 './test-actions/hdf5s/image_expanded_frame_1_305.hdf5',
 './test-actions/hdf5s/image_expanded_frame_2_305.hdf5',
 './test-actions/hdf5s/image_expanded_frame_3_305.hdf5',
 './test-actions/hdf5s/image_expanded_frame_4_305.hdf5',
 './test-actions/hdf5s/image_expanded_frame_5_305.hdf5',
 './test-actions/hdf5s/image_expanded_frame_6_305.hdf5',
 './test-actions/hdf5s/image_expanded_frame_7_305.hdf5',
 './test-actions/hdf5s/image_expanded_frame_8_305.hdf5',
 './test-actions/hdf5s/image_expanded_frame_9_305.hdf5',
 './test-actions/hdf5s/image_expanded_frame_10_305.hdf5',
 './test-actions/hdf5s/image_expanded_frame_11_305.hdf5',
 './test-actions/hdf5s/image_expanded_frame_12_305.hdf5',
 './test-actions/hdf5s/image_expanded_frame_13_305.hdf5',
 './test-actions/hdf5s/image_expanded_frame_14_305.hdf5',
 './test-actions/hdf5s/image_expanded_frame_15_305.hdf5',
 './test-actions/hdf5s/image_expanded_frame_16_305.hdf5',
 './test-actions/hdf5s/i

## Make the pngs

In [12]:
make_plot(0)

image 0 done


In [ ]:
# Just a quick test of the plotting function
make_plot(305)

In [13]:
import multiprocessing
pool = multiprocessing.Pool(8)
tasks = [[ii] for ii in range(len(framenums))]
output = pool.starmap(make_plot, tasks)

image 1 doneimage 0 done

image 3 done
image 2 done
image 5 done
image 7 doneimage 6 done

image 4 done
image 8 doneimage 14 done
image 13 done

image 15 done
image 11 done
image 12 doneimage 10 done

image 9 done
image 17 done
image 18 doneimage 19 doneimage 16 done


image 21 done
image 20 doneimage 22 done



Process ForkPoolWorker-7:
Process ForkPoolWorker-5:
Process ForkPoolWorker-8:
Process ForkPoolWorker-2:
Process ForkPoolWorker-6:
Process ForkPoolWorker-4:
Process ForkPoolWorker-1:
Process ForkPoolWorker-3:


In [ ]:
tasks

## Make movie from pngs

Below a simple ffmpeg command for making the movie.

In [14]:
%%bash
ffmpeg -framerate 8 -i ./../../test-actions/pngs/images_1920_%d.png  -pix_fmt yuv420p ../../lmps_action_movie_8FPS.mp4 -y

ffmpeg version 4.2.9 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 8 (GCC)
  configuration: --prefix=/usr --bindir=/usr/bin --datadir=/usr/share/ffmpeg --docdir=/usr/share/doc/ffmpeg --incdir=/usr/include/ffmpeg --libdir=/usr/lib64 --mandir=/usr/share/man --arch=x86_64 --optflags='-O2 -g -pipe -Wall -Werror=format-security -Wp,-D_FORTIFY_SOURCE=2 -Wp,-D_GLIBCXX_ASSERTIONS -fexceptions -fstack-protector-strong -grecord-gcc-switches -specs=/usr/lib/rpm/redhat/redhat-hardened-cc1 -specs=/usr/lib/rpm/redhat/redhat-annobin-cc1 -m64 -mtune=generic -fasynchronous-unwind-tables -fstack-clash-protection -fcf-protection' --extra-ldflags='-Wl,-z,relro -Wl,-z,now -specs=/usr/lib/rpm/redhat/redhat-hardened-ld ' --extra-cflags=' ' --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libvo-amrwbenc --enable-version3 --enable-bzlib --disable-crystalhd --enable-fontconfig --enable-frei0r --enable-gcrypt --enable-gnutls --enable-ladspa --enable-libaom --enable-libdav1d --enabl

In [15]:
%%bash
ffmpeg -framerate 20 -i ./../../test-actions/pngs/images_1920_%d.png  -pix_fmt yuv420p ../../lmps_action_movie_20FPS.mp4 -y

ffmpeg version 4.2.9 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 8 (GCC)
  configuration: --prefix=/usr --bindir=/usr/bin --datadir=/usr/share/ffmpeg --docdir=/usr/share/doc/ffmpeg --incdir=/usr/include/ffmpeg --libdir=/usr/lib64 --mandir=/usr/share/man --arch=x86_64 --optflags='-O2 -g -pipe -Wall -Werror=format-security -Wp,-D_FORTIFY_SOURCE=2 -Wp,-D_GLIBCXX_ASSERTIONS -fexceptions -fstack-protector-strong -grecord-gcc-switches -specs=/usr/lib/rpm/redhat/redhat-hardened-cc1 -specs=/usr/lib/rpm/redhat/redhat-annobin-cc1 -m64 -mtune=generic -fasynchronous-unwind-tables -fstack-clash-protection -fcf-protection' --extra-ldflags='-Wl,-z,relro -Wl,-z,now -specs=/usr/lib/rpm/redhat/redhat-hardened-ld ' --extra-cflags=' ' --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libvo-amrwbenc --enable-version3 --enable-bzlib --disable-crystalhd --enable-fontconfig --enable-frei0r --enable-gcrypt --enable-gnutls --enable-ladspa --enable-libaom --enable-libdav1d --enabl

## Stitch a 8 FPS video with a 20 FPS video

By playing around, I have found that rotations and zooming in/out looks good with ~20 FPS. Since we normally don't have a lot of snapshots, I've found ~8 FPS works well when changing snapshots (i.e. not too jumpy but also not too fast). Below I have cooked up some code for concatenating a 8 FPS and a 20 FPS movie.

In [ ]:
import subprocess
import os

subprocess.call('rm frame_*.png', shell=True)
for ii in range(138):
    subprocess.call(f'cp ./../../test-actions/pngs/images_1920_{ii}.png frame_{ii}.png', shell=True)
subprocess.call('ffmpeg -framerate 8 -i frame_%d.png -pix_fmt yuv420p input1.mp4 -y', shell=True)
subprocess.call('rm frame_*.png', shell=True)

for jj, ii in enumerate(range(138, len(framenums))):
    subprocess.call(f'cp ./../../test-actions/pngs/images_1920_{ii}.png frame_{jj}.png', shell=True)
subprocess.call('ffmpeg -framerate 20 -i frame_%d.png -pix_fmt yuv420p input2.mp4 -y', shell=True)
subprocess.call('rm frame_*.png', shell=True)
    

In [ ]:
%%bash
ffmpeg -i input1.mp4 -c copy intermediate1.ts -y

In [ ]:
%%bash
ffmpeg -i input2.mp4 -c copy intermediate2.ts -y

In [ ]:
%%bash
ffmpeg -i "concat:intermediate1.ts|intermediate2.ts" -c copy ../../lmps_stitched_action_movie.mp4 -y

## Tracking is all wrong...

In [ ]:
for frame in sorted(framenums):
    im = pa.ImageReader(actions.waypoints[frame])
    print(f'frame={frame}, center={im.center}')